In [1]:
# Import necessary libraries
%matplotlib inline
import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm
import time


In [2]:
# Takes in the next node from the agent and returns the reward
def robot_env(next_node):
    node_rewards = dict()
    with open('nodes_rewards.csv', "r") as nodes_rwrds: #opens the .csv file
        for elements in nodes_rwrds:
            ID_node,Rewards = elements.split(',') 
            node_rewards[ID_node] = Rewards              #assigns ID node to a q_value
        no_of_nodes = int(ID_node)
        #print(no_of_nodes)
    #print(node_rewards[next_node])
    return node_rewards[next_node]

node_rewards = dict()
with open('nodes_rewards.csv', "r") as nodes_rwrds: #opens the .csv file
    for elements in nodes_rwrds:
        ID_node,Rewards = elements.split(',') 
        node_rewards[ID_node] = Rewards              #assigns ID node to a q_value
    no_of_nodes = int(ID_node)
    #print(no_of_nodes)
#print(node_rewards[next_node])

In [3]:
#Adjacency matrix to find the neighbours of a particular node

adj_matrix = np.zeros((no_of_nodes,no_of_nodes))
dist_matrix = np.zeros((no_of_nodes,no_of_nodes))
prob_matrix = np.zeros((no_of_nodes,no_of_nodes))
rwrd_matrix = np.zeros((no_of_nodes,no_of_nodes))

with open('edges.csv', "r") as n_edges: #opens the nodes.csv file
	for elements in n_edges:
		edge_ID1, edge_ID2, distance = elements.split(',') 
		adj_matrix[int(edge_ID1)-1][int(edge_ID2)-1] = 1
		adj_matrix[int(edge_ID2)-1][int(edge_ID1)-1] = 1
		dist_matrix[int(edge_ID1)-1][int(edge_ID2)-1] = distance
		dist_matrix[int(edge_ID2)-1][int(edge_ID1)-1] = distance
        
print(adj_matrix)
print("______________")
print("")

for i in range(no_of_nodes):
    ele_sum = 0
    for j in range(no_of_nodes):
        if adj_matrix[i][j] > 0:
            ele_sum = ele_sum + 1
            
    for k in range(no_of_nodes):
        if adj_matrix[i][k] == 1:
            prob_matrix[i][k] = 1./ele_sum
            
    for l in range(no_of_nodes):
        if adj_matrix[i][k] == 1:
            rwrd_matrix[i][k] = robot_env(str(l+1))
            #print(robot_env(str(l+1)))
    
            
print(prob_matrix)
print("")
print("")

print(rwrd_matrix)            
            
    
    


[[0. 1. 1. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
 [1. 0. 1. 1. 1. 0. 0. 0. 0. 0. 0. 0.]
 [1. 1. 0. 1. 0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 1. 1. 0. 1. 0. 0. 1. 0. 0. 0. 0.]
 [0. 1. 0. 1. 0. 0. 1. 0. 1. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 0. 1. 0. 0. 0.]
 [0. 0. 0. 0. 1. 0. 0. 0. 0. 1. 0. 0.]
 [0. 0. 0. 1. 0. 0. 0. 0. 1. 0. 0. 1.]
 [0. 0. 0. 0. 1. 1. 0. 1. 0. 0. 1. 0.]
 [0. 0. 0. 0. 0. 0. 1. 0. 0. 0. 1. 1.]
 [0. 0. 0. 0. 0. 0. 0. 0. 1. 1. 0. 1.]
 [0. 0. 0. 0. 0. 0. 0. 1. 0. 1. 1. 0.]]
______________

[[0.         0.5        0.5        0.         0.         0.
  0.         0.         0.         0.         0.         0.        ]
 [0.25       0.         0.25       0.25       0.25       0.
  0.         0.         0.         0.         0.         0.        ]
 [0.33333333 0.33333333 0.         0.33333333 0.         0.
  0.         0.         0.         0.         0.         0.        ]
 [0.         0.25       0.25       0.         0.25       0.
  0.         0.25       0.         0.         0.         0.        ]

In [4]:
#Calculate  value functions using Bellman's Equation

v = np.zeros(no_of_nodes)
v_old = v.copy()
gamma = 0.9
delta = 1e-5
delta_t = 1


while delta_t > delta:
    for s in range(len(prob_matrix)):
        v[s] = np.sum([
                prob_matrix[s][sp] * (rwrd_matrix[s][sp] + gamma * v_old[sp])
                for sp in range(len(prob_matrix[s]))
            ])
    delta_t = np.sum(np.abs(v - v_old))
    v_old = v.copy()
    #print(delta_t)
print(v)

[-0.41496071 -0.47423724 -0.44789923 -0.60380179 -0.6410624  -0.78843254
 -0.89509336 -1.12036774 -0.87603693 -1.34803549 -1.34363785 -1.14361163]


In [5]:
def next_start_node(neigh_nodes):
    next_node = 0
    for i in range(len(neigh_nodes)):
        if(v[neigh_nodes[i]-1] > next_node):
            next_node = neigh_nodes[i]
    print(next_node)
    
    return next_node
    

In [6]:
def nearest_nodes(current_node):                       #Find next unexplored nodes
	n_nibor = []
	nibor=0
	i=0
	for i in range(no_of_nodes):
		flag_ok=1
		if adj_matrix[current_node-1][i]==1:
			nibor = i+1
			if nibor in closed:
					flag_ok=0
			if flag_ok and (nibor != current_node):
				n_nibor.append(nibor)

	print ("neighbor --> ", n_nibor)
	return n_nibor

In [7]:
goal_flag = 1
start_node = 1
goal_node = 10
closed = []
path =[]


closed.append(start_node)


while goal_flag:
	path.append(str(start_node))                                #Append paths
	print ("Path  -->  ",path)
	check_nodes = nearest_nodes(start_node)         #Find nearest neighbours of current start node
	start_node = next_start_node(check_nodes)        #Set next node to be explored
	if start_node == goal_node:
		goal_flag = 0
	print ("Next node  -->  ",start_node)
	closed.append(start_node)


Path  -->   ['1']
neighbor -->  [2, 3]
0
Next node  -->   0
Path  -->   ['1', '0']
neighbor -->  [8, 10, 11]
0
Next node  -->   0
Path  -->   ['1', '0', '0']
neighbor -->  [8, 10, 11]
0
Next node  -->   0
Path  -->   ['1', '0', '0', '0']
neighbor -->  [8, 10, 11]
0
Next node  -->   0
Path  -->   ['1', '0', '0', '0', '0']
neighbor -->  [8, 10, 11]
0
Next node  -->   0
Path  -->   ['1', '0', '0', '0', '0', '0']
neighbor -->  [8, 10, 11]
0
Next node  -->   0
Path  -->   ['1', '0', '0', '0', '0', '0', '0']
neighbor -->  [8, 10, 11]
0
Next node  -->   0
Path  -->   ['1', '0', '0', '0', '0', '0', '0', '0']
neighbor -->  [8, 10, 11]
0
Next node  -->   0
Path  -->   ['1', '0', '0', '0', '0', '0', '0', '0', '0']
neighbor -->  [8, 10, 11]
0
Next node  -->   0
Path  -->   ['1', '0', '0', '0', '0', '0', '0', '0', '0', '0']
neighbor -->  [8, 10, 11]
0
Next node  -->   0
Path  -->   ['1', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0']
neighbor -->  [8, 10, 11]
0
Next node  -->   0
Path  -->   ['1

IOPub data rate exceeded.
The notebook server will temporarily stop sending output
to the client in order to avoid crashing it.
To change this limit, set the config variable
`--NotebookApp.iopub_data_rate_limit`.

Current values:
NotebookApp.iopub_data_rate_limit=1000000.0 (bytes/sec)
NotebookApp.rate_limit_window=3.0 (secs)



KeyboardInterrupt: 

In [ ]:
path.append(str(start_node))
#print(adj_matrix[0][3])
#print(adj_matrix)
print ("Path  -->  ",path)
